# Ahmed Mosad 202201982
# Mostafa Mohamed 202201132
# Marwan Amr 202201743

In [ ]:
import numpy as np
from scipy.optimize import line_search
import warnings
import time

# 1D Minimzation

## Fibonacci Method

In [ ]:
def fib1(f, a, b, tol=1e-5, max_iter=100):
    """
    perform fibonacci search to find the minimum of a unimodal function within a specified interval.
    """
    # Generating Fibonacci numbers
    fib = [1, 1]
    while fib[-1] < (b - a) / tol:
        fib.append(fib[-1] + fib[-2])
    
    n = len(fib) - 1
    k = 0
    x1 = a + (fib[n-2] / fib[n]) * (b - a)
    x2 = a + (fib[n-1] / fib[n]) * (b - a)
    f1 = f(x1)
    f2 = f(x2)
    iterations = 0

    while k < n - 2 and iterations < max_iter:
        if f1 > f2:
            a = x1
            x1 = x2
            f1 = f2
            x2 = a + (fib[n-k-1] / fib[n-k]) * (b - a)
            f2 = f(x2)
        else:
            b = x2
            x2 = x1
            f2 = f1
            x1 = a + (fib[n-k-2] / fib[n-k]) * (b - a)
            f1 = f(x1)
        k += 1
        iterations += 1

    if f1 < f2:
        return x1, iterations
    else:
        return x2, iterations


## Golden Section Method

In [ ]:
def golden_section(f, a, b, tol=1e-5, max_iter=100):
    """
    perform golden Section search to find the minimum of a unimodal function within a specified interval.
    """
    phi = (1 + np.sqrt(5)) / 2
    resphi = 2 - phi
    x1 = a + resphi * (b - a)
    x2 = b - resphi * (b - a)
    f1 = f(x1)
    f2 = f(x2)
    iterations = 0

    while abs(b - a) > tol and iterations < max_iter:
        if f1 < f2:
            b = x2
            x2 = x1
            f2 = f1
            x1 = a + resphi * (b - a)
            f1 = f(x1)
        else:
            a = x1
            x1 = x2
            f1 = f2
            x2 = b - resphi * (b - a)
            f2 = f(x2)
        iterations += 1

    if f1 < f2:
        return x1, iterations
    else:
        return x2, iterations


## Newton’s Method

In [ ]:
def newton(df, ddf, x0, tol=1e-5, max_iter=100):
    """
    perform newton's method to find the minimum of a function.
    """
    x = x0
    iterations = 0

    for _ in range(max_iter):
        dfx = df(x)
        ddfx = ddf(x)
        if abs(dfx) < tol:
            break
        x = x - dfx / ddfx
        iterations += 1

    return x, iterations


## Quasi-Newton Method

In [ ]:
def quasi_newton( df, x0, tol=1e-5, max_iter=100):
    """
    perform the quasi-Newton method to find the minimum of a function.
    """
    x = x0
    H = 1.0  # Initial Hessian approximation (scalar for 1D)
    iterations = 0

    for _ in range(max_iter):
        dfx = df(x)
        if abs(dfx) < tol:
            break
        p = -H * dfx
        alpha = 1.0 
        x_new = x + alpha * p
        s = x_new - x
        y = df(x_new) - dfx
        rho = 1.0 / (y * s)
        H = (1 - rho * y * s) * H + rho * s * s
        x = x_new
        iterations += 1

    return x, iterations


## Secant Method

In [ ]:
def secant(df, x0, x1, tol=1e-5, max_iter=100):
    """
    perform  secant method to find the minimum of a function.
    """
    iterations = 0

    for _ in range(max_iter):
        dfx0 = df(x0)
        dfx1 = df(x1)
        if abs(dfx1) < tol:
            break
        x2 = x1 - dfx1 * (x1 - x0) / (dfx1 - dfx0)
        x0, x1 = x1, x2
        iterations += 1

    return x1, iterations


# Unconstrained nonlinear optimization 

## Fletcher-Reeves CG Method

In [ ]:
def fletcher_reeves(f, grad_f, x0, tol=1e-6, max_iter=1000):
    x = np.array(x0, dtype=float)
    g = grad_f(x)
    d = -g
    iterations = 0
    fx = f(x)
    
    while iterations < max_iter:
        try:
            alpha = line_search(f, grad_f, x, d, c1=1e-4, c2=0.1, maxiter=100)[0]
        except:
            alpha = 0.1 / (1 + iterations)
        
        if alpha is None:
            alpha = 0.1 / (1 + iterations)
            
        x_new = x + alpha * d
        g_new = grad_f(x_new)
        fx_new = f(x_new)
        
        if np.linalg.norm(g_new) < tol or abs(fx_new - fx) < tol:
            x = x_new
            break
            
        beta = np.dot(g_new, g_new) / max(np.dot(g, g), 1e-10)
        if iterations % 10 == 0 or beta < 0:
            d = -g_new
        else:
            d = -g_new + beta * d
            
        x = x_new
        g = g_new
        fx = fx_new
        iterations += 1
        
        if not np.all(np.isfinite(x)) or not np.all(np.isfinite(d)):
            warnings.warn("numerical issues found")
            break
            
    return x, iterations, f(x)


## Marquardt Method

In [ ]:
def marquardt(f, grad_f, hess_f, x0, tol=1e-6, max_iter=1000, lambda_=1.0):
    x = np.array(x0, dtype=float)
    iterations = 0
    
    while np.linalg.norm(grad_f(x)) > tol and iterations < max_iter:
        try:
            g = grad_f(x)
            H = hess_f(x)
            H_lm = H + lambda_ * np.eye(len(x))
            
            if np.linalg.cond(H_lm) > 1e10:
                H_lm += 1e-6 * np.eye(len(x))
                
            dx = np.linalg.solve(H_lm, -g)
            x_new = x + dx
            
            if f(x_new) < f(x):
                x = x_new
                lambda_ = max(lambda_/10, 1e-7)
            else:
                lambda_ = min(lambda_*10, 1e7)
                
            iterations += 1
            
        except np.linalg.LinAlgError:
            warnings.warn("linear algebra error..........")
            break
            
    return x, iterations, f(x)


## Quasi-Newton Method

In [ ]:
def quasi_newton(f, grad_f, x0, tol=1e-6, max_iter=1000):
    x = np.array(x0, dtype=float)
    n = len(x)
    B = np.eye(n)
    iterations = 0
    
    while np.linalg.norm(grad_f(x)) > tol and iterations < max_iter:
        try:
            g = grad_f(x)
            d = -np.linalg.solve(B, g)
            alpha = line_search(f, grad_f, x, d, c1=1e-4, c2=0.9)[0]
            if alpha is None:
                alpha = 1.0 / max(2**iterations, 1e-10)
                
            s = alpha * d
            x_new = x + s
            y = grad_f(x_new) - g
            
            sy = np.dot(s, y)
            if sy > 1e-10:
                Bs = B @ s
                B = B - np.outer(Bs, Bs)/np.dot(s, Bs) + np.outer(y, y)/sy
                
            x = x_new
            iterations += 1
            
        except Exception as e:
            warnings.warn(f"optimization not succeed : {str(e)}")
            break
            
    return x, iterations, f(x)


# Problems

## Rosenbrock’s Parabolic Valley Function

In [ ]:


def rosenbrock(x):
    return 100*(x[1] - x[0]**2)**2 + (1-x[0])**2



def rosenbrock_grad(x):
    return np.array([
        -400*x[0]*(x[1] - x[0]**2) - 2*(1-x[0]),
        200*(x[1] - x[0]**2)
    ])


def rosenbrock_hess(x):
    return np.array([
        [-400*(x[1] - 3*x[0]**2) + 2, -400*x[0]],
        [-400*x[0], 200]
    ])

## Powell’s Quartic Function

In [ ]:

def powell(x):
    return (x[0] + 10*x[1])**2 + 5*(x[2] - x[3])**2 + (x[1] - 2*x[2])**4 + 10*(x[0] - x[3])**4

def powell_grad(x):
    return np.array([
        2*(x[0] + 10*x[1]) + 40*(x[0] - x[3])**3,
        20*(x[0] + 10*x[1]) + 4*(x[1] - 2*x[2])**3,
        -8*(x[1] - 2*x[2])**3 + 10*(x[2] - x[3]),
        -10*(x[2] - x[3]) - 40*(x[0] - x[3])**3
    ])

def powell_hess(x):
    return np.array([
        [2 + 120*(x[0] - x[3])**2, 20, 0, -120*(x[0] - x[3])**2],
        [20, 200 + 12*(x[1] - 2*x[2])**2, -24*(x[1] - 2*x[2])**2, 0],
        [0, -24*(x[1] - 2*x[2])**2, 10 + 48*(x[1] - 2*x[2])**2, -10],
        [-120*(x[0] - x[3])**2, 0, -10, 10 + 120*(x[0] - x[3])**2]
    ])

# Comparing the algorithms

In [ ]:
def compare_methods(name, f, grad_f, hess_f, x0):
    methods = [
        ("Fletcher-Reeves", lambda x0: fletcher_reeves(f, grad_f, x0)),
        ("Marquardt", lambda x0: marquardt(f, grad_f, hess_f, x0)),
        ("Quasi-Newton", lambda x0: quasi_newton(f, grad_f, x0))
    ]
    
    print(f"\n{name} Results:")
    print("=" * 50)
    
    for method_name, method in methods:
        start_time = time.time()
        x_opt, iterations, f_opt = method(x0)
        elapsed_time = time.time() - start_time
        
        print(f"\n{method_name}:")
        print(f"Number of iterations : -> {iterations}")
        print(f"The optimal solution.: -> {x_opt}")
        print(f"The optimal value.: -> {f_opt}")
        print(f"CPU time.:-> {elapsed_time:.6f} seconds")


# Output

In [ ]:
x0_rosenbrock = np.array([-1.2, 1.0])
x0_powell = np.array([3.0, -1.0, 0.0, 1.0])

compare_methods("Rosenbrock Function", rosenbrock, rosenbrock_grad, rosenbrock_hess, x0_rosenbrock)
compare_methods("Powell Function", powell, powell_grad, powell_hess, x0_powell)